In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import joblib
import copy
from collections import Counter

from dotenv import load_dotenv
load_dotenv()

# Add parent directory to path
import sys
sys.path.append("../..")

from findai.entities import ThreadedConversation, Message

# Collection of original communication: 

In [ ]:
# read json fine by path: 
import json
import pandas as pd
import numpy as np
import os
import re
from collections import Counter

from tqdm import tqdm

# get all directories in the current directory:

channels_ids = []

telegram_path = "../../data/telegram/01_raw_data"
directories = [d for d in os.listdir(telegram_path) if os.path.isdir(os.path.join(telegram_path, d))]

stats_all = []

for directory in tqdm(directories):
    data_file = os.path.join(telegram_path, directory, "result.json")
    # Load the JSON file: 
    with open(data_file, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    messages = data['messages']
    
    # Calculate statistics: 
    total_messages = len(messages)
    unique_users, min_time, max_time, message_lenght, is_service_message, is_channel_post = [], [], [], [], [], []
    for msg in tqdm(messages):
        if 'from_id' in msg:
            unique_users.append(msg['from_id'])
            if 'channel' in msg['from_id']:
                is_channel_post.append(1)
        if 'date' in msg:
            min_time.append(msg['date'])
            max_time.append(msg['date'])
        if 'text' in msg:
            if isinstance(msg['text'], list):
                message_lenght.append(len("".join([i["text"] if isinstance(i, dict) else i for i in msg['text'] if 'text' in i])))
            elif isinstance(msg['text'], str):
                message_lenght.append(len(msg['text']))
            else:
                message_lenght.append(0)
                print("Error: ", msg['text'])
                print(type(msg['text']))
        if "type" in msg:
            if msg['type'] == 'service':
                is_service_message.append(1)
    
    channels_messages = [m.get("from_id") for m in messages if m is not None and "channel" in m.get("from_id", "")]
    most_common_channel = Counter(channels_messages).most_common(1)[0][0]

    unique_users = len(set(unique_users))
    min_time = min(min_time)
    max_time = max(max_time)
    message_lenght_mean = sum(message_lenght) / len(message_lenght) if message_lenght else 0
    message_lenght_median = sorted(message_lenght)[len(message_lenght) // 2] if message_lenght else 0
    message_lenght_max = max(message_lenght) if message_lenght else 0
    message_lenght_min = min(message_lenght) if message_lenght else 0
    number_of_messages_with_text = np.sum([1 for i in message_lenght if i > 0])
    number_of_service_messages = np.sum([1 for i in is_service_message if i == 1])
    number_of_channel_posts = np.sum([1 for i in is_channel_post if i == 1])

    stats_dict = {
        'channel': directory,
        'channel_id': most_common_channel,
        'total_messages': total_messages,
        'unique_users': unique_users,
        'min_time': min_time,
        'max_time': max_time,
        'message_lenght_mean': message_lenght_mean,
        'message_lenght_median': message_lenght_median,
        'message_lenght_max': message_lenght_max,
        'message_lenght_min': message_lenght_min,
        'number_of_messages_with_text': number_of_messages_with_text,
        'number_of_service_messages': number_of_service_messages,
        'number_of_channel_posts': number_of_channel_posts
    }

    stats_all.append(stats_dict)

In [ ]:
print(pd.DataFrame(stats_all).to_markdown(index=False))

## Collect all usernames for proper anonymization: 

In [ ]:
# Function for user anonymization:
import random
import string

def generate_random_str_id(length=8, existing_ids={}):
    """
    Generates a random string ID of a given length.
    Args:
        length (int): Length of the random string ID.
        existing_ids (set): A dictionary of existing IDs to avoid duplicates.
    Returns:
        str: A unique random string ID.
    """
    
    while True:
        random_id = ''.join(random.choices(string.ascii_letters + string.digits, k=length))
        if random_id not in existing_ids:
            return random_id

def anonymize_user_id(user_id, user_id_mapping, ids_used):
    """
    Anonymizes the user ID by replacing it with a unique identifier based on a mapping.
    Args:
        user_id (str): The original user ID.
        user_id_mapping (dict): A mapping of original user IDs to anonymized IDs.
    Returns:
        str: The anonymized user ID.
    """
    if user_id not in user_id_mapping:
        # Generate a new random ID for the user:
        anonymized_id = generate_random_str_id(existing_ids=ids_used)
        user_id_mapping[user_id] = anonymized_id
        ids_used.add(anonymized_id)
    return user_id_mapping[user_id]

## Processing messages with anonimization: 

In [ ]:
# def collect_msg_type_examples(messages):
#     examples = {}
    
#     def scan(text_field):
#         if isinstance(text_field, list):
#             for item in text_field:
#                 scan(item)
#         elif isinstance(text_field, dict):
#             msg_type = text_field.get("type", "")
#             if msg_type and msg_type not in examples:
#                 examples[msg_type] = text_field
#             if "text" in text_field:
#                 scan(text_field["text"])
    
#     for m in messages:
#         if "text" in m:
#             scan(m["text"])
    
#     return examples

# # Run across all channels:
# all_examples = {}
# for channel_stat in tqdm(stats_all):
#     channel = channel_stat['channel']
#     data_file = os.path.join(telegram_path, channel, "result.json")
#     with open(data_file, 'r', encoding='utf-8') as f:
#         data = json.load(f)
#     new_examples = collect_msg_type_examples(data['messages'])
#     # Only add types not yet seen:
#     for msg_type, example in new_examples.items():
#         if msg_type not in all_examples:
#             all_examples[msg_type] = example

# for msg_type, example in all_examples.items():
#     print(f"\n--- {msg_type} ---")
#     print(example)


In [ ]:
from urllib.parse import urlparse

def collect_channel_usernames(messages):
    usernames = set()

    def scan(text_field):
        if isinstance(text_field, list):
            for item in text_field:
                scan(item)
        elif isinstance(text_field, dict):
            msg_type = text_field.get("type", "").lower()
            if msg_type == "mention":
                username = text_field.get("text", "").lstrip("@")
                if username:
                    usernames.add(username)
            elif msg_type == "mention_name":
                user_id = str(text_field.get("user_id", ""))
                if user_id:
                    usernames.add(user_id)
                display_name = text_field.get("text", "")
                if display_name:
                    usernames.add(display_name)
            if "text" in text_field:
                scan(text_field["text"])

    for m in messages:
        if "from_id" in m:
            usernames.add(m["from_id"])
        if "from" in m and m["from"]:
            usernames.add(str(m["from"]))
        for field in ("text", "text_entities"):
            if field in m:
                scan(m[field])
    return usernames


def extract_telegram_message_text(text_field, user_id_mapping=None, ids_used=None):
    if isinstance(text_field, list):
        return "".join([extract_telegram_message_text(item, user_id_mapping, ids_used) for item in text_field])

    elif isinstance(text_field, dict):
        txt = extract_telegram_message_text(text_field["text"], user_id_mapping, ids_used) if "text" in text_field else str(text_field)
        msg_type = text_field.get("type", "").lower()

        if msg_type == "mention":
            username = txt.lstrip("@")
            anon = anonymize_user_id(username, user_id_mapping, ids_used) if user_id_mapping is not None else "[USER]"
            txt = f"@{anon}"
        elif msg_type == "mention_name":
            key = str(text_field.get("user_id", txt))
            anon = anonymize_user_id(key, user_id_mapping, ids_used) if user_id_mapping is not None else "[USER]"
            txt = f"@{anon}"
        elif msg_type == "phone":
            txt = "[PHONE]"
        elif msg_type == "bank_card":
            txt = "[BANK_CARD]"
        elif msg_type == "email":
            txt = "[EMAIL]"
        elif msg_type == "link":
            try:
                parsed = urlparse(txt if txt.startswith("http") else f"https://{txt}")
                txt = f"{parsed.scheme}://{parsed.netloc}" if parsed.netloc else "[LINK]"
            except Exception as e:
                txt = "[LINK]"
                print(e)
        elif msg_type == "bot_command":
            txt = re.sub(r'@\w+', '@[BOT]', txt)
        elif msg_type == "bold":
            txt = f"**{txt}**"
        elif msg_type == "italic":
            txt = f"*{txt}*"
        elif msg_type == "underline":
            txt = f"__{txt}__"
        elif msg_type == "strikethrough":
            txt = f"~~{txt}~~"
        elif msg_type == "spoiler":
            txt = f"||{txt}||"
        elif msg_type == "blockquote":
            txt = f"> {txt}"
        elif msg_type in ("code", "pre"):
            txt = f"`{txt}`"
        elif msg_type == "text_link":
            pass
        return txt
    else:
        return str(text_field)

In [ ]:
all_threads = {}
all_user_names = {}

for channel_stat in tqdm(stats_all):
    channel_id = channel_stat['channel_id']
    channel = channel_stat['channel']
    data_file = os.path.join(telegram_path, channel, "result.json")
    print(data_file)

    with open(data_file, 'r', encoding='utf-8') as file:
        data = json.load(file)

    messages = data['messages']
    print(len(messages), " messages in the channel: ", channel)

    # build per-channel anonymization mapping
    user_id_mapping = {}
    ids_used = set()
    for username in collect_channel_usernames(messages):
        anonymize_user_id(username, user_id_mapping, ids_used)
    all_user_names[channel] = user_id_mapping

    # build threads with anonymization applied
    all_threads_channel = {}
    message_thread_mapping = {}   
    message_id_mapping = {}
    thread_counter = 0

    for m in tqdm(messages, total=len(messages)):
        if m['type'] == 'service':
            pass
        if m.get('from_id', "") == channel_id and m.get('reply_to_message_id') is None:
            anon_thread_id = f"thread_{thread_counter}"
            thread_counter += 1

            thread = ThreadedConversation(
                id=anon_thread_id,
                initial_post=extract_telegram_message_text(m.get('text', ""), user_id_mapping, ids_used),
                previous_comments=[]
            )
            all_threads_channel[anon_thread_id] = thread
            message_thread_mapping[m['id']] = anon_thread_id

        else:
            if "reply_to_message_id" not in m:
                continue

            raw_reply_id = m["reply_to_message_id"]
            original_thread_key = f"{channel_id}_{raw_reply_id}"

            if raw_reply_id in message_thread_mapping:
                anon_thread_id = message_thread_mapping[raw_reply_id]
                thread = all_threads_channel[anon_thread_id]
                anon_reply_id = message_id_mapping.get(raw_reply_id)  # reply to a comment
            elif any(t.id == f"{channel_id}_{raw_reply_id}" for t in all_threads_channel.values()):
                continue
            else:
                continue

            msg_index = len(thread.previous_comments)
            anon_msg_id = f"{anon_thread_id}_msg_{msg_index}"
            message_id_mapping[m['id']] = anon_msg_id
            message_thread_mapping[m['id']] = anon_thread_id

            message = Message(
                id=anon_msg_id,
                message_text=extract_telegram_message_text(m.get('text', ""), user_id_mapping, ids_used),
                user_id=anonymize_user_id(m['from_id'], user_id_mapping, ids_used),
                answer_to_message_id=anon_reply_id,
                timestamp=m.get('date'),
                metadata={}
            )
            if "[USER]" in message.message_text:
                print(message)
            thread.previous_comments.append(message)
            all_threads_channel[anon_thread_id] = thread

    all_threads[channel] = all_threads_channel

# Checking the anonymization quality: 

In [ ]:
# all_text_per_channel = {
#     channel: " ".join(
#         message.message_text
#         for thread in threads.values()
#         for message in thread.previous_comments
#     )
#     for channel, threads in all_threads.items()
# }

# leaks_per_channel = {}

# for channel, all_text in all_text_per_channel.items():
#     print("-"*30 + channel)

#     original_ids = set(all_user_names[channel].keys())
    
#     # One pass to find all @mentions in text
#     found_mentions = set(re.findall(r'@(\w+)\b', all_text))
#     leaked = found_mentions & original_ids
#     leaked = set([s for s in leaked if len(s) > 3])
#     print(leaked)
#     print(f"{channel}: {len(leaked)} leaked")

#     if leaked:
#         leak_examples = {}
#         for thread in all_threads[channel].values():
#             for message in thread.previous_comments:
#                 if len(leak_examples) == len(leaked):
#                     break
#                 for uid in leaked:
#                     if uid not in leak_examples and f"@{uid}" in message.message_text:
#                         leak_examples[uid] = message.message_text
#         leaks_per_channel[channel] = leak_examples
    
#     for uid, text in leaks_per_channel.get(channel, {}).items():
#         print(f"@{uid}  →  {text}\n")

In [ ]:
# Fix in-place (if any)
for channel, leak_examples in leaks_per_channel.items():
    leaked_ids = set(leak_examples.keys())
    user_id_mapping = all_user_names[channel]
    ids_used = set(user_id_mapping.values())
    
    for thread in all_threads[channel].values():
        for message in thread.previous_comments:
            for uid in leaked_ids:
                if f"@{uid}" in message.message_text:
                    anon = anonymize_user_id(uid, user_id_mapping, ids_used)
                    message.message_text = message.message_text.replace(f"@{uid}", f"@{anon}")

# Filtering and saving the data: 

In [ ]:
all_threads_filtered = {}
for channel, all_threads_channel in all_threads.items():
    # Count threads per user
    user_thread_counter = Counter()
    for conversation in all_threads_channel.values():
        users_in_thread = set(message.user_id for message in conversation.previous_comments)
        for user_id in users_in_thread:
            user_thread_counter[user_id] += 1

    # Keep users with at least 10 threads
    eligible_users = set(uid for uid, count in user_thread_counter.items() if count >= 10)
    print(f"{channel}: {len(eligible_users)} eligible users")

    # Filter threads that have at least one eligible user
    all_threads_filtered[channel] = [
        conversation for conversation in all_threads_channel.values()
        if any(message.user_id in eligible_users for message in conversation.previous_comments)
    ]

In [ ]:
# Save
joblib.dump(all_threads_filtered, "../../data/telegram/02_processed_data/telegram_threads_filtered.joblib")
print("Number of channels: ", len(all_threads_filtered))
joblib.dump(all_threads, "../../data/telegram/02_processed_data/telegram_threads_all.joblib")
print("Number of channels: ", len(all_threads))